# Yaw Walk With Me - an IMU-Based Head Tracking for Augmented Auditory Reality implemented on the Tiresias project

<p align="center">
    <img src="figures/tiresias-icon-full.png" width="450">
</p>

This notebook presents a real-time head tracking system designed for immersive audio and Augmented Auditory Reality (AAR) applications.

The implementation is based on the open-source hearing research platform <a href="https://tiresias-web-gamma.vercel.app">Tiresias</a>, employing:

- Bosch BMI270 IMU
- Nordic nRF5340 platform
- BLE quaternion streaming (GATT Server)
- Python-based visualization (GATT Client)
- Real-time spatial interaction

The proposed framework investigates how user head orientation can be incorporated into immersive auditory environments for interaction with Virtual Sound Objects (VSOs), enabling attention-driven selection mechanisms for spatial audio systems.

The architecture combines:

- embedded sensing
- quaternion orientation estimation
- low-latency BLE communication
- 3D rendering
- head-referenced calibration
- angular interaction logic

This proof-of-concept aligns with Augmented Auditory Reality concepts proposed for ecologically valid hearing assistance systems and immersive auditory interaction (Grimm et al., 2018; Mehra et al., 2020), and can be illustrated as follows:

<p align="center">
    <img src="figures/objective.png" width="800">
</p>

## Required Libraries

The implementation combines asynchronous BLE communication, quaternion processing, and real-time 3D rendering.

The following libraries are employed:

| Library | Purpose |
|---|---|
| `bleak` | BLE communication |
| `numpy` | Numerical processing |
| `PyQt6` | GUI framework |
| `pyqtgraph.opengl` | Real-time 3D visualization |
| `numpy-stl` | STL mesh loading |
| `qasync` | Asyncio + Qt event loop integration |

In [14]:
import sys
print(sys.executable)

!{sys.executable} -m pip -q install numpy bleak nest_asyncio asyncio ipympl pyqtgraph PyQt6 PyOpenGL

import asyncio
import struct
import math
import nest_asyncio
import numpy as np

from bleak import BleakScanner, BleakClient
from qasync import QEventLoop

import pyqtgraph.opengl as gl

from stl import mesh

from PyQt6.QtWidgets import QApplication
from PyQt6.QtCore import QTimer
from PyQt6.QtGui import QMatrix4x4

nest_asyncio.apply()

/Users/joaovitor/Documents/PhD/aulas_USP/VR-AR/yaw-walk-with-me/venv/bin/python

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Firmware Architecture

The embedded firmware was developed on top of the Zephyr RTOS and organized into a modular subsystem architecture available at the <a href="https://github.com/felipepimentab/tiresias-fw">Tiresias Firmware Page</a>.

The architecture separates functionality into three abstraction levels — Application, Module, and Driver layers — enabling portability, scalability, and simplified integration.

<p align="center">
    <img src="figures/fw.png" width="900">
</p>

The Application Layer implements high-level functionalities such as BLE communication, head tracking, audio configuration, and user interaction.

The Module Layer encapsulates reusable services including orientation estimation, parameter adjustment, storage management, and communication interfaces.

The Driver Layer abstracts hardware-specific peripherals such as the BMI270 IMU, audio codec interfaces, GPIOs, I2C, and I2S communication buses.

For this work the IMU application subsystem was implemented together with its **head tracking** module and dedicate **Bosch drivers interface**, as well as a **Orientation** BLE module.   

## BLE Communication Architecture

The proposed head tracking framework employs Bluetooth Low Energy (BLE) as a low-latency communication interface between the embedded platform and the Python-based immersive audio application.

The embedded firmware periodically publishes **quaternion orientation estimates obtained from the BMI270 IMU through a custom BLE GATT service**. On the host side, the Python application scans nearby BLE devices, identifies the Tiresias platform, and subscribes to orientation notifications in real time.

The BLE subsystem was implemented using Zephyr Bluetooth APIs and custom Generic Attribute Profile (GATT) characteristics specifically designed for quaternion streaming.

In [15]:
import asyncio
from bleak import BleakScanner

async def scan():
    devices = await BleakScanner.discover(
        return_adv=True,
        timeout=10
    )

    for addr, (device, adv) in devices.items():
        print("\n================")
        print("ADDRESS:", device.address)
        print("NAME:", device.name)
        print("RSSI:", adv.rssi)
        print("LOCAL NAME:", adv.local_name)
        print("UUIDS:", adv.service_uuids)
        print("MFG DATA:", adv.manufacturer_data)

await scan()


ADDRESS: 5FAAE2EC-74EB-08C3-6EF0-DA068E957342
NAME: iPhone de Joao victor (3)
RSSI: -57
LOCAL NAME: None
UUIDS: []
MFG DATA: {76: b'\x10\x06=\x1e8\x1b\x16\xef'}

ADDRESS: C04C97BA-ED53-E435-DDE4-7055A36EF461
NAME: Tiresias_DK
RSSI: -69
LOCAL NAME: Tiresias_DK
UUIDS: ['00001523-1212-efde-1523-785feabcd123']
MFG DATA: {}

ADDRESS: C2C172FC-98D0-5509-9DEE-45912D1D5EB4
NAME: None
RSSI: -58
LOCAL NAME: None
UUIDS: []
MFG DATA: {76: b'\x12\x02\x00\x03'}

ADDRESS: 21E58A86-DA67-E46D-4CB8-FB28BD892221
NAME: None
RSSI: -57
LOCAL NAME: None
UUIDS: []
MFG DATA: {76: b'\x16\x08\x00>\x8dbn\xc41\x10'}


## IMU Orientation Estimation Pipeline

The proposed head tracking system estimates orientation through the fusion of inertial sensor measurements obtained from the BMI270 IMU:

$$
\text{Accelerometer} + \text{Gyroscope}
\rightarrow
\text{Sensor Fusion}
\rightarrow
\text{Quaternion}
\rightarrow
\text{Euler Angles}
$$

### Accelerometer

The accelerometer measures linear acceleration along the three spatial axes:

$$
(a_x, a_y, a_z)
$$

Under static or low-dynamic conditions, gravity provides a reference for estimating inclination relative to the Earth.

However, accelerometers alone cannot reliably estimate **yaw orientation**.

### Gyroscope

The gyroscope measures angular velocity:

$$
(\omega_x, \omega_y, \omega_z)
$$

By integrating angular velocity over time, rotational motion can be estimated with high temporal resolution.

Nevertheless, **gyroscope integration accumulates drift over time** due to sensor bias and noise.

### Sensor Fusion

To overcome the limitations of individual sensors, sensor fusion algorithms are employied combining:

- short-term rotational precision from the gyroscope
- long-term gravitational reference from the accelerometer

The resulting estimate provides stable and continuous orientation tracking suitable for immersive interaction.

### Quaternion Representation

The fused orientation estimate is internally represented as a quaternion:

$$
q = (q_w, q_x, q_y, q_z)
$$

Quaternions efficiently encode 3D rotations while avoiding singularities such as gimbal lock.

The quaternion stream is transmitted through BLE to the host application.

### Euler Angles

For interaction logic and interpretation, the quaternion is converted into Euler angles:

- Roll
- Pitch
- Yaw

In the proposed framework, the yaw angle is employed as the primary interaction variable for spatial source selection and user intent estimation.

<p align="center">
    <img src="figures/euler.jpg" width="400">
</p>

In [16]:
## The consumer can be implemented directly from the main application script 

!python main.py

Scanning for BLE devices...

Found device: Tiresias_DK
Address: C04C97BA-ED53-E435-DDE4-7055A36EF461

Connecting...

Connected.
Receiving head orientation...


Forward direction calibrated.


Who killed Laura Palmer??


Fire walk with me



## Current Status and Future Work

The proposed framework successfully demonstrates:

- real-time IMU-based head tracking
- BLE quaternion streaming
- quaternion-to-Euler conversion
- head-referenced calibration
- immersive 3D visualization
- orientation-driven interaction logic

The current implementation already enables directional interaction through head orientation, where left/right angular displacement triggers interaction events associated with spatial attention.

At the present stage, the interaction mechanism is represented through **logging events intended only to validate the proposed interaction model**.

Future work will focus on replacing the current logging-based proof-of-concept with a complete auditory attention framework capable of:

- selecting Virtual Sound Objects (VSOs)
- dynamically controlling source gain
- spatially rendering target sources
- incorporating multimodal attention estimation

In particular, a more **advanced attention model** should be incorporated to better represent user intent and auditory scene interaction.

Additionally, the current implementation does not yet perform dynamic binaural rendering using Head-Related Transfer Functions (HRTFs).

For perceptually coherent immersive audio reproduction, **HRTF processing must be performed continuously and frame-by-frame while updating source positions in real time according to head orientation**.

This dynamic spatial rendering stage is fundamental to preserving stable localization cues and maintaining perceptual consistency between user motion and the virtual auditory scene.